# Deepfake Detection - Week 3 CNN Baseline (ResNet-50)

This Colab notebook trains a frame-level ResNet-50 baseline with transfer learning, early stopping, learning-rate scheduling, and test-set evaluation.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import sys

drive.mount('/content/drive')

# Update this path if your folder location is different in Drive.
PROJECT_ROOT = Path('/content/drive/MyDrive/deepfake-detection')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}')

sys.path.append(str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

In [ ]:
# Optional: if dataset is not already in Drive, copy from Kaggle here.
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # Upload kaggle.json if needed
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d <your-dataset-id> -p /content/
# !unzip -q /content/<archive>.zip -d {PROJECT_ROOT}

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from src.data.dataset import build_dataloaders
from src.models.resnet_classifier import create_model_and_optimizer

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 2  # Colab can usually handle 2-4
IMAGE_SIZE = 224
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.4
TRAINABLE_BACKBONE_LAYERS = 1
NUM_EPOCHS = 20
PATIENCE = 3
GRAD_ACCUM_STEPS = 1

MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(
    project_root=PROJECT_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    image_size=IMAGE_SIZE,
    max_frames_per_video=None,
    max_train_samples=None,
    max_val_samples=None,
    max_test_samples=None,
)

print(f'Train samples: {len(train_loader.dataset)}')
print(f'Val samples  : {len(val_loader.dataset)}')
print(f'Test samples : {len(test_loader.dataset)}')

In [ ]:
model, criterion, optimizer = create_model_and_optimizer(
    device=device,
    pretrained=True,
    dropout=DROPOUT,
    trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=1
)

checkpoint_path = MODELS_DIR / 'cnn_baseline_best.pth'
history_path = OUTPUTS_DIR / 'train_history_colab.json'
curves_path = OUTPUTS_DIR / 'training_curves.png'

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, grad_accum_steps=1):
    model.train()
    running_loss, running_acc, total_batches = 0.0, 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for batch_idx, (images, labels) in enumerate(tqdm(loader, desc='Train', leave=False)):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().view(-1)

        logits = model(images)
        loss = criterion(logits, labels)
        (loss / grad_accum_steps).backward()

        if (batch_idx + 1) % grad_accum_steps == 0 or (batch_idx + 1) == len(loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            acc = (preds == labels).float().mean().item()

        running_loss += loss.item()
        running_acc += acc
        total_batches += 1

    return running_loss / max(total_batches, 1), running_acc / max(total_batches, 1)


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, running_acc, total_batches = 0.0, 0.0, 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val', leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).float().view(-1)

            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            acc = (preds == labels).float().mean().item()

            running_loss += loss.item()
            running_acc += acc
            total_batches += 1

    return running_loss / max(total_batches, 1), running_acc / max(total_batches, 1)


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, NUM_EPOCHS + 1):
    print(f'\nEpoch {epoch}/{NUM_EPOCHS}')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, GRAD_ACCUM_STEPS)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)

    print(f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} | lr={lr:.6f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
                'best_val_loss': min(best_val_loss, val_loss),
                'history': history,
            },
            checkpoint_path,
        )
        print('Saved best model:', checkpoint_path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping triggered (patience={PATIENCE}).')
            break

with open(history_path, 'w', encoding='utf-8') as fp:
    json.dump(history, fp, indent=2)

print('\nBest validation accuracy:', f'{best_val_acc:.4f}')

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], label='Train Loss')
plt.plot(epochs, history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], label='Train Accuracy')
plt.plot(epochs, history['val_acc'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curves')
plt.legend()

plt.tight_layout()
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved curves to:', curves_path)

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_targets, all_preds, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Test', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).float().view(-1)

        logits = model(images)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        all_targets.extend(labels.long().cpu().tolist())
        all_preds.extend(preds.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

y_true = np.array(all_targets)
y_pred = np.array(all_preds)

test_accuracy = accuracy_score(y_true, y_pred)
test_precision = precision_score(y_true, y_pred, zero_division=0)
test_recall = recall_score(y_true, y_pred, zero_division=0)
test_f1 = f1_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['real', 'fake'], digits=4, zero_division=0)

print(f'Accuracy : {test_accuracy:.4f}')
print(f'Precision: {test_precision:.4f}')
print(f'Recall   : {test_recall:.4f}')
print(f'F1-Score : {test_f1:.4f}')
print('\nClassification Report:\n')
print(report)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Real', 'Pred Fake'], yticklabels=['True Real', 'True Fake'])
plt.title('CNN Baseline - Test Confusion Matrix')
plt.tight_layout()
cm_path = OUTPUTS_DIR / 'cnn_confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion matrix to:', cm_path)

In [ ]:
report_path = OUTPUTS_DIR / 'cnn_evaluation.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('CNN BASELINE EVALUATION REPORT\n')
    f.write('=' * 40 + '\n')
    f.write(f'Checkpoint: {checkpoint_path}\n')
    f.write(f'Accuracy : {test_accuracy:.4f}\n')
    f.write(f'Precision: {test_precision:.4f}\n')
    f.write(f'Recall   : {test_recall:.4f}\n')
    f.write(f'F1-Score : {test_f1:.4f}\n\n')
    f.write('Confusion Matrix [[TN, FP], [FN, TP]]:\n')
    f.write(str(cm.tolist()) + '\n\n')
    f.write('Classification Report:\n')
    f.write(report + '\n')

print('Saved evaluation report to:', report_path)
print('Best model path:', checkpoint_path)
files.download(str(checkpoint_path))